# 12 — Order Fulfillment Analysis

## Sales Order, Backlog & Inventory Analytics

### Objective

Analyze the end-to-end order fulfillment process from customer order placement through quantity fulfillment, shipment, and final delivery performance.

### Analysis Scope

This notebook will focus on:

- Order fulfillment performance
- Ordered vs fulfilled quantities
- Backlog impact on fulfillment
- Shipment coverage of fulfilled orders
- End-to-end order flow
- Fulfillment by priority
- Fulfillment by sales channel
- Fulfillment by customer segment
- Fulfillment by region
- Fulfillment by warehouse
- Fulfillment by product category
- Order-to-shipment timing
- Delivery performance
- OTIF performance
- Fulfillment and delivery relationship
- High-risk fulfillment patterns
- End-to-end service performance

### Goal

The objective is to determine where customer orders experience service failures across the complete fulfillment lifecycle and identify the operational areas that have the greatest impact on customer service performance.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

print(
    "\nOrder Fulfillment Analysis environment initialized successfully."
)

Pandas version : 3.0.6
NumPy version  : 2.5.3

Order Fulfillment Analysis environment initialized successfully.


## Step 2 — Define Feature-Engineered Data Path

Define the project directory and the location of the feature-engineered datasets.

The Order Fulfillment Analysis will use the existing analysis-ready data to maintain consistency with the completed Sales, Backlog, Shipment & Delivery, and Inventory analyses.

In [2]:
# Define project paths

project_path = Path.cwd().parent

featured_data_path = (
    project_path / "data" / "featured"
)

print("Project Path       :", project_path)
print("Featured Data Path :", featured_data_path)

print(
    "\nFeatured Data Folder Exists :",
    featured_data_path.exists()
)

Project Path       : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics
Featured Data Path : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics\data\featured

Featured Data Folder Exists : True


## Step 3 — Load Order Fulfillment Datasets

Load the core feature-engineered datasets required for end-to-end order fulfillment analysis.

The analysis uses:

- **Sales Orders** — order-level fulfillment, backlog, customer, priority, channel, and warehouse information
- **Sales Order Items** — product-level ordered, fulfilled, and backlog quantities
- **Shipments** — shipment timing, delivery performance, and OTIF results

These datasets will later be connected to analyze the complete order-to-delivery lifecycle.

In [4]:
# Load Order Fulfillment datasets

sales_orders = pd.read_csv(
    featured_data_path / "sales_orders_featured.csv",
    parse_dates=[
        "order_date",
        "requested_delivery_date"
    ]
)

sales_order_items = pd.read_csv(
    featured_data_path / "sales_order_items_featured.csv",
    parse_dates=["order_date"]
)

shipments = pd.read_csv(
    featured_data_path / "shipments_featured.csv",
    parse_dates=[
        "shipment_date",
        "requested_delivery_date",
        "actual_delivery_date"
    ]
)


fulfillment_datasets = {
    "Sales Orders": sales_orders,
    "Sales Order Items": sales_order_items,
    "Shipments": shipments
}


print("Order Fulfillment datasets loaded successfully.\n")


for name, df in fulfillment_datasets.items():

    print(
        f"{name:<20} "
        f"{df.shape[0]:>7,} rows × "
        f"{df.shape[1]:>2} columns"
    )

Order Fulfillment datasets loaded successfully.

Sales Orders          25,000 rows × 26 columns
Sales Order Items     61,397 rows × 20 columns
Shipments             24,798 rows × 21 columns


## Step 4 — Validate Required Fulfillment Fields

Validate the fields required for the end-to-end order fulfillment analysis.

The validation covers:

- Order quantities and values
- Fulfilled and backlog quantities
- Order status and fill rate
- Priority, channel, and warehouse
- Product-level fulfillment
- Shipment quantities
- Shipment and delivery dates
- Clean delivery-performance indicators
- OTIF status

All required fields must be available before the order, item, and shipment datasets are integrated.

In [5]:
# Define required columns

required_columns = {

    "Sales Orders": [
        "order_id",
        "customer_id",
        "order_date",
        "warehouse_id",
        "priority",
        "sales_channel",
        "requested_delivery_date",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty",
        "order_value",
        "backlog_value",
        "order_fill_rate_pct",
        "order_status"
    ],

    "Sales Order Items": [
        "order_item_id",
        "order_id",
        "product_id",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty",
        "line_fill_rate_pct",
        "backlog_status"
    ],

    "Shipments": [
        "shipment_id",
        "order_id",
        "warehouse_id",
        "shipment_date",
        "requested_delivery_date",
        "actual_delivery_date",
        "shipped_qty",
        "chronology_valid_flag",
        "delivery_delay_days_clean",
        "on_time_flag_clean",
        "otif_flag_clean"
    ]
}


validation_results = {}


for name, columns in required_columns.items():

    df = fulfillment_datasets[name]

    missing_columns = [
        col for col in columns
        if col not in df.columns
    ]

    validation_results[name] = (
        len(missing_columns) == 0
    )

    print(f"\n{name}")
    print("-" * 45)

    print(
        "Required Fields :",
        len(columns)
    )

    print(
        "Missing Fields  :",
        len(missing_columns)
    )

    if missing_columns:
        print(
            "Missing Columns :",
            missing_columns
        )
    else:
        print(
            "Status          : PASSED"
        )


all_fields_valid = all(
    validation_results.values()
)


print(
    "\nREQUIRED FIELD VALIDATION :",
    "PASSED"
    if all_fields_valid
    else "REVIEW REQUIRED"
)


Sales Orders
---------------------------------------------
Required Fields : 14
Missing Fields  : 0
Status          : PASSED

Sales Order Items
---------------------------------------------
Required Fields : 8
Missing Fields  : 0
Status          : PASSED

Shipments
---------------------------------------------
Required Fields : 11
Missing Fields  : 0
Status          : PASSED

REQUIRED FIELD VALIDATION : PASSED


## Step 5 — Validate Order-Level Keys Before Integration

Before integrating order and shipment information, validate the uniqueness and coverage of `order_id`.

This step determines:

- Whether Sales Orders contains one record per order
- Whether Shipments contains one or multiple records per order
- How many orders have shipment records
- How many orders currently have no shipment record
- Whether any shipment references an unknown order

Understanding the data grain before merging prevents incorrect order counts and duplicated business metrics.

In [5]:
# Validate order-level keys and shipment coverage

total_orders = len(sales_orders)

unique_order_ids = sales_orders["order_id"].nunique()

duplicate_order_ids = sales_orders[
    "order_id"
].duplicated().sum()


shipment_records = len(shipments)

unique_shipped_orders = shipments[
    "order_id"
].nunique()

duplicate_shipment_order_ids = shipments[
    "order_id"
].duplicated().sum()


orders_with_shipment = sales_orders[
    "order_id"
].isin(
    shipments["order_id"]
).sum()

orders_without_shipment = (
    total_orders - orders_with_shipment
)


unknown_shipment_orders = (
    ~shipments["order_id"].isin(
        sales_orders["order_id"]
    )
).sum()


print("ORDER & SHIPMENT KEY VALIDATION\n")

print(
    f"Total Sales Orders                 : {total_orders:,}"
)

print(
    f"Unique Sales Order IDs             : {unique_order_ids:,}"
)

print(
    f"Duplicate Sales Order IDs          : {duplicate_order_ids:,}"
)

print()

print(
    f"Total Shipment Records             : {shipment_records:,}"
)

print(
    f"Unique Orders in Shipments         : {unique_shipped_orders:,}"
)

print(
    f"Repeated Order IDs in Shipments    : {duplicate_shipment_order_ids:,}"
)

print()

print(
    f"Orders With Shipment               : {orders_with_shipment:,}"
)

print(
    f"Orders Without Shipment            : {orders_without_shipment:,}"
)

print(
    f"Shipment Records With Unknown Order: {unknown_shipment_orders:,}"
)

ORDER & SHIPMENT KEY VALIDATION

Total Sales Orders                 : 25,000
Unique Sales Order IDs             : 25,000
Duplicate Sales Order IDs          : 0

Total Shipment Records             : 24,798
Unique Orders in Shipments         : 24,798
Repeated Order IDs in Shipments    : 0

Orders With Shipment               : 24,798
Orders Without Shipment            : 202
Shipment Records With Unknown Order: 0


In [6]:
# Select shipment fields required for order-level analysis

shipment_order_data = shipments[
    [
        "order_id",
        "shipment_id",
        "shipment_date",
        "actual_delivery_date",
        "shipped_qty",
        "chronology_valid_flag",
        "delivery_delay_days_clean",
        "on_time_flag_clean",
        "otif_flag_clean"
    ]
].copy()


# Merge Sales Orders with Shipment information

order_fulfillment = sales_orders.merge(
    shipment_order_data,
    on="order_id",
    how="left",
    validate="one_to_one"
)


# Create shipment availability flag

order_fulfillment["shipment_status"] = np.where(
    order_fulfillment["shipment_id"].notna(),
    "Shipped",
    "Not Shipped"
)


print("ORDER FULFILLMENT BASE CREATED\n")

print(
    f"Order Fulfillment Records : {len(order_fulfillment):,}"
)

print(
    f"Unique Orders             : "
    f"{order_fulfillment['order_id'].nunique():,}"
)

print(
    f"Shipped Orders            : "
    f"{(order_fulfillment['shipment_status'] == 'Shipped').sum():,}"
)

print(
    f"Orders Without Shipment   : "
    f"{(order_fulfillment['shipment_status'] == 'Not Shipped').sum():,}"
)

print(
    f"Duplicate Order IDs       : "
    f"{order_fulfillment['order_id'].duplicated().sum():,}"
)

print(
    f"\nOrder Fulfillment Shape   : "
    f"{order_fulfillment.shape}"
)

ORDER FULFILLMENT BASE CREATED

Order Fulfillment Records : 25,000
Unique Orders             : 25,000
Shipped Orders            : 24,798
Orders Without Shipment   : 202
Duplicate Order IDs       : 0

Order Fulfillment Shape   : (25000, 35)


## Step 7 — Overall Order Fulfillment Performance

Calculate the overall order fulfillment performance across all customer orders.

Key metrics include:

- Total Orders
- Total Ordered Quantity
- Total Fulfilled Quantity
- Total Backlog Quantity
- Overall Quantity Fulfillment Rate
- Orders Fully Fulfilled
- Orders with Backlog
- Average Order Fill Rate

These KPIs establish the baseline service performance before examining fulfillment differences across operational dimensions.

In [7]:
# Overall Order Fulfillment KPIs

total_orders = order_fulfillment["order_id"].nunique()

total_ordered_qty = order_fulfillment[
    "ordered_qty"
].sum()

total_fulfilled_qty = order_fulfillment[
    "fulfilled_qty"
].sum()

total_backlog_qty = order_fulfillment[
    "backlog_qty"
].sum()


overall_fulfillment_rate = (
    total_fulfilled_qty
    / total_ordered_qty
    * 100
)


fully_fulfilled_orders = (
    order_fulfillment["backlog_qty"] == 0
).sum()


orders_with_backlog = (
    order_fulfillment["backlog_qty"] > 0
).sum()


fully_fulfilled_order_rate = (
    fully_fulfilled_orders
    / total_orders
    * 100
)


backlog_order_rate = (
    orders_with_backlog
    / total_orders
    * 100
)


average_order_fill_rate = order_fulfillment[
    "order_fill_rate_pct"
].mean()


print("OVERALL ORDER FULFILLMENT PERFORMANCE\n")

print(
    f"Total Orders                  : {total_orders:,}"
)

print(
    f"Total Ordered Quantity        : {total_ordered_qty:,.0f}"
)

print(
    f"Total Fulfilled Quantity      : {total_fulfilled_qty:,.0f}"
)

print(
    f"Total Backlog Quantity        : {total_backlog_qty:,.0f}"
)

print(
    f"Overall Quantity Fill Rate    : {overall_fulfillment_rate:.2f}%"
)

print()

print(
    f"Fully Fulfilled Orders        : {fully_fulfilled_orders:,}"
)

print(
    f"Fully Fulfilled Order Rate    : {fully_fulfilled_order_rate:.2f}%"
)

print(
    f"Orders With Backlog           : {orders_with_backlog:,}"
)

print(
    f"Backlog Order Rate            : {backlog_order_rate:.2f}%"
)

print()

print(
    f"Average Order Fill Rate       : {average_order_fill_rate:.2f}%"
)

OVERALL ORDER FULFILLMENT PERFORMANCE

Total Orders                  : 25,000
Total Ordered Quantity        : 501,755
Total Fulfilled Quantity      : 440,576
Total Backlog Quantity        : 61,179
Overall Quantity Fill Rate    : 87.81%

Fully Fulfilled Orders        : 14,150
Fully Fulfilled Order Rate    : 56.60%
Orders With Backlog           : 10,850
Backlog Order Rate            : 43.40%

Average Order Fill Rate       : 87.44%


## Step 8 — Fulfillment Performance by Order Priority

Analyze order fulfillment performance across different order-priority levels.

The analysis compares:

- Total Orders
- Ordered Quantity
- Fulfilled Quantity
- Backlog Quantity
- Quantity Fill Rate
- Average Order Fill Rate
- Orders with Backlog
- Backlog Order Rate

This helps determine whether higher-priority customer orders are receiving better fulfillment service or experiencing greater supply constraints.

In [7]:
# Select shipment fields required for order-level analysis

shipment_order_data = shipments[
    [
        "order_id",
        "shipment_id",
        "shipment_date",
        "actual_delivery_date",
        "shipped_qty",
        "chronology_valid_flag",
        "delivery_delay_days_clean",
        "on_time_flag_clean",
        "otif_flag_clean"
    ]
].copy()


# Merge Sales Orders with Shipment information

order_fulfillment = sales_orders.merge(
    shipment_order_data,
    on="order_id",
    how="left",
    validate="one_to_one"
)


# Create shipment availability flag

order_fulfillment["shipment_status"] = np.where(
    order_fulfillment["shipment_id"].notna(),
    "Shipped",
    "Not Shipped"
)


print("ORDER FULFILLMENT BASE CREATED\n")

print(
    f"Order Fulfillment Records : {len(order_fulfillment):,}"
)

print(
    f"Unique Orders             : "
    f"{order_fulfillment['order_id'].nunique():,}"
)

print(
    f"Shipped Orders            : "
    f"{(order_fulfillment['shipment_status'] == 'Shipped').sum():,}"
)

print(
    f"Orders Without Shipment   : "
    f"{(order_fulfillment['shipment_status'] == 'Not Shipped').sum():,}"
)

print(
    f"Duplicate Order IDs       : "
    f"{order_fulfillment['order_id'].duplicated().sum():,}"
)

print(
    f"\nOrder Fulfillment Shape   : "
    f"{order_fulfillment.shape}"
)

ORDER FULFILLMENT BASE CREATED

Order Fulfillment Records : 25,000
Unique Orders             : 25,000
Shipped Orders            : 24,798
Orders Without Shipment   : 202
Duplicate Order IDs       : 0

Order Fulfillment Shape   : (25000, 35)


In [8]:
# Fulfillment performance by order priority

priority_fulfillment = (
    order_fulfillment
    .groupby("priority", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        avg_order_fill_rate_pct=("order_fill_rate_pct", "mean")
    )
)


# Calculate quantity-level fill rate

priority_fulfillment["quantity_fill_rate_pct"] = (
    priority_fulfillment["fulfilled_qty"]
    / priority_fulfillment["ordered_qty"]
    * 100
)


# Count orders with backlog by priority

priority_backlog_orders = (
    order_fulfillment[
        order_fulfillment["backlog_qty"] > 0
    ]
    .groupby("priority")
    ["order_id"]
    .nunique()
)


priority_fulfillment["orders_with_backlog"] = (
    priority_fulfillment["priority"]
    .map(priority_backlog_orders)
    .fillna(0)
    .astype(int)
)


# Calculate backlog order rate

priority_fulfillment["backlog_order_rate_pct"] = (
    priority_fulfillment["orders_with_backlog"]
    / priority_fulfillment["total_orders"]
    * 100
)


# Arrange columns

priority_fulfillment = priority_fulfillment[
    [
        "priority",
        "total_orders",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty",
        "quantity_fill_rate_pct",
        "avg_order_fill_rate_pct",
        "orders_with_backlog",
        "backlog_order_rate_pct"
    ]
]


# Sort from strongest to weakest quantity fill rate

priority_fulfillment = (
    priority_fulfillment
    .sort_values(
        "quantity_fill_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("FULFILLMENT PERFORMANCE BY ORDER PRIORITY\n")

print(
    priority_fulfillment.to_string(
        index=False,
        formatters={
            "quantity_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "avg_order_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "backlog_order_rate_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

FULFILLMENT PERFORMANCE BY ORDER PRIORITY

priority  total_orders  ordered_qty  fulfilled_qty  backlog_qty quantity_fill_rate_pct avg_order_fill_rate_pct  orders_with_backlog backlog_order_rate_pct
    High          6359       129220         114007        15213                 88.23%                  87.53%                 2735                 43.01%
  Normal         14557       292757         257101        35656                 87.82%                  87.39%                 6385                 43.86%
     Low          2035        39688          34600         5088                 87.18%                  87.54%                  854                 41.97%
  Urgent          2049        40090          34868         5222                 86.97%                  87.48%                  876                 42.75%


## Step 9 — Fulfillment Performance by Sales Channel

Analyze fulfillment performance across sales channels.

The analysis compares:

- Total Orders
- Ordered Quantity
- Fulfilled Quantity
- Backlog Quantity
- Quantity Fill Rate
- Average Order Fill Rate
- Orders with Backlog
- Backlog Order Rate

This helps determine whether any sales channel experiences systematically weaker order fulfillment performance.

In [9]:
# Fulfillment performance by sales channel

channel_fulfillment = (
    order_fulfillment
    .groupby("sales_channel", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        avg_order_fill_rate_pct=("order_fill_rate_pct", "mean")
    )
)


# Quantity-level fulfillment rate

channel_fulfillment["quantity_fill_rate_pct"] = (
    channel_fulfillment["fulfilled_qty"]
    / channel_fulfillment["ordered_qty"]
    * 100
)


# Orders containing backlog

channel_backlog_orders = (
    order_fulfillment[
        order_fulfillment["backlog_qty"] > 0
    ]
    .groupby("sales_channel")["order_id"]
    .nunique()
)


channel_fulfillment["orders_with_backlog"] = (
    channel_fulfillment["sales_channel"]
    .map(channel_backlog_orders)
    .fillna(0)
    .astype(int)
)


# Backlog order rate

channel_fulfillment["backlog_order_rate_pct"] = (
    channel_fulfillment["orders_with_backlog"]
    / channel_fulfillment["total_orders"]
    * 100
)


# Arrange columns

channel_fulfillment = channel_fulfillment[
    [
        "sales_channel",
        "total_orders",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty",
        "quantity_fill_rate_pct",
        "avg_order_fill_rate_pct",
        "orders_with_backlog",
        "backlog_order_rate_pct"
    ]
]


# Strongest to weakest quantity fill rate

channel_fulfillment = (
    channel_fulfillment
    .sort_values(
        "quantity_fill_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("FULFILLMENT PERFORMANCE BY SALES CHANNEL\n")

print(
    channel_fulfillment.to_string(
        index=False,
        formatters={
            "quantity_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "avg_order_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "backlog_order_rate_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

FULFILLMENT PERFORMANCE BY SALES CHANNEL

sales_channel  total_orders  ordered_qty  fulfilled_qty  backlog_qty quantity_fill_rate_pct avg_order_fill_rate_pct  orders_with_backlog backlog_order_rate_pct
  Distributor          5469       109514          96900        12614                 88.48%                  87.83%                 2362                 43.19%
       Direct          7195       146393         128435        17958                 87.73%                  87.46%                 3136                 43.59%
       Online          7851       157738         138112        19626                 87.56%                  87.09%                 3406                 43.38%
  Marketplace          4485        88110          77129        10981                 87.54%                  87.57%                 1946                 43.39%


## Step 10 — Fulfillment Performance by Customer Segment

Analyze whether fulfillment performance differs across customer segments.

Customer attributes are added from the feature-engineered customer dimension using `customer_id`.

The analysis compares:

- Total Orders
- Ordered Quantity
- Fulfilled Quantity
- Backlog Quantity
- Quantity Fill Rate
- Average Order Fill Rate
- Orders with Backlog
- Backlog Order Rate

This helps identify whether specific customer groups experience weaker fulfillment service.

In [10]:
# Load customer dimension

customers = pd.read_csv(
    featured_data_path / "customers_featured.csv"
)


# Add customer segment to order fulfillment base

order_fulfillment = order_fulfillment.merge(
    customers[
        [
            "customer_id",
            "customer_segment"
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)


# Validate customer matches

print(
    "Missing Customer Segment After Merge :",
    order_fulfillment["customer_segment"].isna().sum()
)


# Fulfillment performance by customer segment

segment_fulfillment = (
    order_fulfillment
    .groupby("customer_segment", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        avg_order_fill_rate_pct=("order_fill_rate_pct", "mean")
    )
)


segment_fulfillment["quantity_fill_rate_pct"] = (
    segment_fulfillment["fulfilled_qty"]
    / segment_fulfillment["ordered_qty"]
    * 100
)


segment_backlog_orders = (
    order_fulfillment[
        order_fulfillment["backlog_qty"] > 0
    ]
    .groupby("customer_segment")["order_id"]
    .nunique()
)


segment_fulfillment["orders_with_backlog"] = (
    segment_fulfillment["customer_segment"]
    .map(segment_backlog_orders)
    .fillna(0)
    .astype(int)
)


segment_fulfillment["backlog_order_rate_pct"] = (
    segment_fulfillment["orders_with_backlog"]
    / segment_fulfillment["total_orders"]
    * 100
)


segment_fulfillment = (
    segment_fulfillment
    .sort_values(
        "quantity_fill_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nFULFILLMENT PERFORMANCE BY CUSTOMER SEGMENT\n")

print(
    segment_fulfillment.to_string(
        index=False,
        formatters={
            "quantity_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "avg_order_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "backlog_order_rate_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

Missing Customer Segment After Merge : 0

FULFILLMENT PERFORMANCE BY CUSTOMER SEGMENT

customer_segment  total_orders  ordered_qty  fulfilled_qty  backlog_qty avg_order_fill_rate_pct quantity_fill_rate_pct  orders_with_backlog backlog_order_rate_pct
     Distributor          3577        71872          63417         8455                  87.77%                 88.24%                 1528                 42.72%
          Retail          9269       186425         163763        22662                  87.36%                 87.84%                 4049                 43.68%
             SMB          7856       157154         137813        19341                  87.46%                 87.69%                 3405                 43.34%
      Enterprise          4298        86304          75583        10721                  87.34%                 87.58%                 1868                 43.46%


## Step 11 — Fulfillment Performance by Customer Region

Analyze order fulfillment performance across customer regions.

The analysis compares:

- Total Orders
- Ordered Quantity
- Fulfilled Quantity
- Backlog Quantity
- Quantity Fill Rate
- Average Order Fill Rate
- Orders with Backlog
- Backlog Order Rate

This helps identify geographic areas where customers may experience weaker fulfillment performance.

In [11]:
# Add customer region to the order fulfillment base

customer_region = customers[
    [
        "customer_id",
        "region"
    ]
].rename(
    columns={
        "region": "customer_region"
    }
)


order_fulfillment = order_fulfillment.merge(
    customer_region,
    on="customer_id",
    how="left",
    validate="many_to_one"
)


print(
    "Missing Customer Region After Merge :",
    order_fulfillment["customer_region"].isna().sum()
)


# Fulfillment performance by customer region

region_fulfillment = (
    order_fulfillment
    .groupby("customer_region", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        avg_order_fill_rate_pct=("order_fill_rate_pct", "mean")
    )
)


region_fulfillment["quantity_fill_rate_pct"] = (
    region_fulfillment["fulfilled_qty"]
    / region_fulfillment["ordered_qty"]
    * 100
)


region_backlog_orders = (
    order_fulfillment[
        order_fulfillment["backlog_qty"] > 0
    ]
    .groupby("customer_region")["order_id"]
    .nunique()
)


region_fulfillment["orders_with_backlog"] = (
    region_fulfillment["customer_region"]
    .map(region_backlog_orders)
    .fillna(0)
    .astype(int)
)


region_fulfillment["backlog_order_rate_pct"] = (
    region_fulfillment["orders_with_backlog"]
    / region_fulfillment["total_orders"]
    * 100
)


region_fulfillment = (
    region_fulfillment
    .sort_values(
        "quantity_fill_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nFULFILLMENT PERFORMANCE BY CUSTOMER REGION\n")

print(
    region_fulfillment.to_string(
        index=False,
        formatters={
            "quantity_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "avg_order_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "backlog_order_rate_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

Missing Customer Region After Merge : 0

FULFILLMENT PERFORMANCE BY CUSTOMER REGION

customer_region  total_orders  ordered_qty  fulfilled_qty  backlog_qty avg_order_fill_rate_pct quantity_fill_rate_pct  orders_with_backlog backlog_order_rate_pct
           West          6198       123862         109543        14319                  87.70%                 88.44%                 2647                 42.71%
           East          3906        78534          69139         9395                  87.91%                 88.04%                 1637                 41.91%
          South          6995       140654         123803        16851                  87.42%                 88.02%                 3061                 43.76%
        Central          3160        61992          54049         7943                  86.99%                 87.19%                 1391                 44.02%
          North          4741        96713          84042        12671                  87.08%           

## Step 12 — Fulfillment Performance by Warehouse

Analyze order fulfillment performance across fulfillment warehouses.

The analysis compares:

- Total Orders
- Ordered Quantity
- Fulfilled Quantity
- Backlog Quantity
- Quantity Fill Rate
- Average Order Fill Rate
- Orders with Backlog
- Backlog Order Rate

Warehouse-level analysis helps identify operational locations that may be contributing disproportionately to fulfillment shortages and backlog.

In [12]:
# Load warehouse dimension

warehouses = pd.read_csv(
    featured_data_path / "warehouses_featured.csv"
)


# Create warehouse lookup

warehouse_lookup = warehouses[
    [
        "warehouse_id",
        "warehouse_name"
    ]
].copy()


# Fulfillment performance by warehouse

warehouse_fulfillment = (
    order_fulfillment
    .groupby("warehouse_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        avg_order_fill_rate_pct=("order_fill_rate_pct", "mean")
    )
)


# Add warehouse name

warehouse_fulfillment = warehouse_fulfillment.merge(
    warehouse_lookup,
    on="warehouse_id",
    how="left",
    validate="one_to_one"
)


# Quantity fill rate

warehouse_fulfillment["quantity_fill_rate_pct"] = (
    warehouse_fulfillment["fulfilled_qty"]
    / warehouse_fulfillment["ordered_qty"]
    * 100
)


# Orders with backlog

warehouse_backlog_orders = (
    order_fulfillment[
        order_fulfillment["backlog_qty"] > 0
    ]
    .groupby("warehouse_id")["order_id"]
    .nunique()
)


warehouse_fulfillment["orders_with_backlog"] = (
    warehouse_fulfillment["warehouse_id"]
    .map(warehouse_backlog_orders)
    .fillna(0)
    .astype(int)
)


# Backlog order rate

warehouse_fulfillment["backlog_order_rate_pct"] = (
    warehouse_fulfillment["orders_with_backlog"]
    / warehouse_fulfillment["total_orders"]
    * 100
)


# Arrange columns

warehouse_fulfillment = warehouse_fulfillment[
    [
        "warehouse_id",
        "warehouse_name",
        "total_orders",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty",
        "quantity_fill_rate_pct",
        "avg_order_fill_rate_pct",
        "orders_with_backlog",
        "backlog_order_rate_pct"
    ]
]


# Sort strongest to weakest fill rate

warehouse_fulfillment = (
    warehouse_fulfillment
    .sort_values(
        "quantity_fill_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("FULFILLMENT PERFORMANCE BY WAREHOUSE\n")

print(
    warehouse_fulfillment.to_string(
        index=False,
        formatters={
            "quantity_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "avg_order_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "backlog_order_rate_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

FULFILLMENT PERFORMANCE BY WAREHOUSE

warehouse_id    warehouse_name  total_orders  ordered_qty  fulfilled_qty  backlog_qty quantity_fill_rate_pct avg_order_fill_rate_pct  orders_with_backlog backlog_order_rate_pct
        WH02       Mumbai West          4441        89564          79058        10506                 88.27%                  87.65%                 1923                 43.30%
        WH03         Delhi NCR          4812        96611          85205        11406                 88.19%                  87.61%                 2058                 42.77%
        WH01 Bengaluru Central          5457       109658          96452        13206                 87.96%                  87.56%                 2372                 43.47%
        WH04   Hyderabad South          3566        70048          61484         8564                 87.77%                  87.22%                 1535                 43.05%
        WH06           Pune DC          3266        65847          57368     

## Step 13 — Fulfillment Performance by Product Category

Analyze product-level fulfillment performance across product categories.

Sales Order Items are connected with the Product dimension to evaluate:

- Number of Order Lines
- Ordered Quantity
- Fulfilled Quantity
- Backlog Quantity
- Quantity Fill Rate
- Average Line Fill Rate
- Backlogged Order Lines
- Backlog Line Rate

This identifies product categories that contribute disproportionately to fulfillment shortages.

In [13]:
# Load product dimension

products = pd.read_csv(
    featured_data_path / "products_featured.csv"
)


# Add product category to Sales Order Items

item_fulfillment = sales_order_items.merge(
    products[
        [
            "product_id",
            "product_name",
            "category"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)


print(
    "Missing Product Category After Merge :",
    item_fulfillment["category"].isna().sum()
)


# Fulfillment performance by product category

category_fulfillment = (
    item_fulfillment
    .groupby("category", as_index=False)
    .agg(
        total_order_lines=("order_item_id", "nunique"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        avg_line_fill_rate_pct=("line_fill_rate_pct", "mean")
    )
)


# Quantity fill rate

category_fulfillment["quantity_fill_rate_pct"] = (
    category_fulfillment["fulfilled_qty"]
    / category_fulfillment["ordered_qty"]
    * 100
)


# Backlogged order lines

category_backlog_lines = (
    item_fulfillment[
        item_fulfillment["backlog_qty"] > 0
    ]
    .groupby("category")["order_item_id"]
    .nunique()
)


category_fulfillment["backlogged_order_lines"] = (
    category_fulfillment["category"]
    .map(category_backlog_lines)
    .fillna(0)
    .astype(int)
)


# Backlog line rate

category_fulfillment["backlog_line_rate_pct"] = (
    category_fulfillment["backlogged_order_lines"]
    / category_fulfillment["total_order_lines"]
    * 100
)


# Arrange columns

category_fulfillment = category_fulfillment[
    [
        "category",
        "total_order_lines",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty",
        "quantity_fill_rate_pct",
        "avg_line_fill_rate_pct",
        "backlogged_order_lines",
        "backlog_line_rate_pct"
    ]
]


# Sort strongest to weakest quantity fill rate

category_fulfillment = (
    category_fulfillment
    .sort_values(
        "quantity_fill_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nFULFILLMENT PERFORMANCE BY PRODUCT CATEGORY\n")

print(
    category_fulfillment.to_string(
        index=False,
        formatters={
            "quantity_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "avg_line_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "backlog_line_rate_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

Missing Product Category After Merge : 0

FULFILLMENT PERFORMANCE BY PRODUCT CATEGORY

              category  total_order_lines  ordered_qty  fulfilled_qty  backlog_qty quantity_fill_rate_pct avg_line_fill_rate_pct  backlogged_order_lines backlog_line_rate_pct
        Home & Kitchen              11584        95924          85219        10705                 88.84%                 88.28%                    2335                20.16%
       Office Supplies              11401        92730          81623        11107                 88.02%                 87.70%                    2376                20.84%
           Electronics              16347       134143         117919        16224                 87.91%                 87.02%                    3620                22.14%
Health & Personal Care              11161        90438          79126        11312                 87.49%                 87.05%                    2438                21.84%
            Automotive              10

## Step 14 — Shipment Coverage vs Order Fulfillment Status

Analyze whether orders with different fulfillment outcomes successfully progressed to shipment.

Orders are classified into:

- Fully Fulfilled
- Partially Fulfilled / Backlogged

The analysis compares shipment coverage for each group.

This helps distinguish inventory fulfillment problems from downstream shipment-execution problems.

In [14]:
# Create order fulfillment status

order_fulfillment["fulfillment_status"] = np.where(
    order_fulfillment["backlog_qty"] == 0,
    "Fully Fulfilled",
    "Partially Fulfilled / Backlogged"
)


# Fulfillment status vs shipment status

fulfillment_shipment = (
    order_fulfillment
    .groupby(
        [
            "fulfillment_status",
            "shipment_status"
        ],
        as_index=False
    )
    .agg(
        total_orders=("order_id", "nunique"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum")
    )
)


# Total orders within each fulfillment status

fulfillment_status_totals = (
    fulfillment_shipment
    .groupby("fulfillment_status")["total_orders"]
    .transform("sum")
)


# Shipment-status share within fulfillment group

fulfillment_shipment["order_share_pct"] = (
    fulfillment_shipment["total_orders"]
    / fulfillment_status_totals
    * 100
)


print("SHIPMENT COVERAGE VS ORDER FULFILLMENT STATUS\n")

print(
    fulfillment_shipment.to_string(
        index=False,
        formatters={
            "order_share_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


# Overall shipment coverage

shipment_coverage_rate = (
    (
        order_fulfillment["shipment_status"]
        == "Shipped"
    ).mean()
    * 100
)


print(
    f"\nOverall Shipment Coverage : "
    f"{shipment_coverage_rate:.2f}%"
)

SHIPMENT COVERAGE VS ORDER FULFILLMENT STATUS

              fulfillment_status shipment_status  total_orders  ordered_qty  fulfilled_qty  backlog_qty order_share_pct
                 Fully Fulfilled         Shipped         14150       260352         260352            0         100.00%
Partially Fulfilled / Backlogged     Not Shipped           202         1671              0         1671           1.86%
Partially Fulfilled / Backlogged         Shipped         10648       239732         180224        59508          98.14%

Overall Shipment Coverage : 99.19%


## Step 15 — Validate Shipment Quantity Against Fulfilled Quantity

Compare order-level fulfilled quantity with shipment quantity.

This analysis determines whether:

- Fulfilled quantities are fully represented in shipment records
- Shipment quantities are lower than fulfilled quantities
- Shipment quantities exceed recorded fulfilled quantities
- Quantity differences exist between fulfillment and shipment execution

This is an important reconciliation step in the end-to-end order lifecycle.

In [15]:
# Compare fulfilled quantity with shipped quantity

order_fulfillment["shipped_qty_filled"] = (
    order_fulfillment["shipped_qty"]
    .fillna(0)
)


order_fulfillment["fulfilled_vs_shipped_diff"] = (
    order_fulfillment["fulfilled_qty"]
    - order_fulfillment["shipped_qty_filled"]
)


# Classify quantity relationship

conditions = [
    order_fulfillment["fulfilled_vs_shipped_diff"] == 0,
    order_fulfillment["fulfilled_vs_shipped_diff"] > 0,
    order_fulfillment["fulfilled_vs_shipped_diff"] < 0
]

labels = [
    "Matched",
    "Fulfilled > Shipped",
    "Shipped > Fulfilled"
]


order_fulfillment["fulfillment_shipment_qty_status"] = (
    np.select(
        conditions,
        labels,
        default="Review"
    )
)


# Summary

quantity_reconciliation = (
    order_fulfillment
    .groupby(
        "fulfillment_shipment_qty_status",
        as_index=False
    )
    .agg(
        total_orders=("order_id", "nunique"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        shipped_qty=("shipped_qty_filled", "sum"),
        quantity_difference=(
            "fulfilled_vs_shipped_diff",
            "sum"
        )
    )
)


quantity_reconciliation["order_share_pct"] = (
    quantity_reconciliation["total_orders"]
    / len(order_fulfillment)
    * 100
)


print("FULFILLED QUANTITY VS SHIPPED QUANTITY\n")

print(
    quantity_reconciliation.to_string(
        index=False,
        formatters={
            "order_share_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


print(
    "\nTotal Fulfilled Quantity :",
    f"{order_fulfillment['fulfilled_qty'].sum():,.0f}"
)

print(
    "Total Shipped Quantity   :",
    f"{order_fulfillment['shipped_qty_filled'].sum():,.0f}"
)

print(
    "Net Quantity Difference  :",
    f"{order_fulfillment['fulfilled_vs_shipped_diff'].sum():,.0f}"
)

FULFILLED QUANTITY VS SHIPPED QUANTITY

fulfillment_shipment_qty_status  total_orders  fulfilled_qty  shipped_qty  quantity_difference order_share_pct
                        Matched         25000         440576     440576.0                  0.0         100.00%

Total Fulfilled Quantity : 440,576
Total Shipped Quantity   : 440,576
Net Quantity Difference  : 0


## Step 16 — Order-to-Shipment Processing Time

Measure the elapsed time between order placement and shipment.

This analysis evaluates:

- Average Order-to-Shipment Days
- Median Order-to-Shipment Days
- Minimum and Maximum Processing Time
- Processing-Time Distribution

Understanding order-to-shipment time helps identify how quickly fulfilled customer demand moves through warehouse execution.

In [16]:
# Calculate order-to-shipment processing time

shipped_orders = order_fulfillment[
    order_fulfillment["shipment_status"] == "Shipped"
].copy()


shipped_orders["order_to_shipment_days"] = (
    shipped_orders["shipment_date"]
    - shipped_orders["order_date"]
).dt.days


print("ORDER-TO-SHIPMENT PROCESSING TIME\n")

print(
    f"Shipped Orders              : "
    f"{len(shipped_orders):,}"
)

print(
    f"Average Processing Days     : "
    f"{shipped_orders['order_to_shipment_days'].mean():.2f}"
)

print(
    f"Median Processing Days      : "
    f"{shipped_orders['order_to_shipment_days'].median():.2f}"
)

print(
    f"Minimum Processing Days     : "
    f"{shipped_orders['order_to_shipment_days'].min():.0f}"
)

print(
    f"Maximum Processing Days     : "
    f"{shipped_orders['order_to_shipment_days'].max():.0f}"
)


# Create processing-time bands

shipped_orders["processing_time_band"] = pd.cut(
    shipped_orders["order_to_shipment_days"],
    bins=[
        -np.inf,
        0,
        1,
        2,
        3,
        5,
        np.inf
    ],
    labels=[
        "Same Day",
        "1 Day",
        "2 Days",
        "3 Days",
        "4-5 Days",
        "6+ Days"
    ]
)


processing_distribution = (
    shipped_orders
    .groupby(
        "processing_time_band",
        observed=False
    )
    .agg(
        total_orders=("order_id", "nunique")
    )
    .reset_index()
)


processing_distribution["order_share_pct"] = (
    processing_distribution["total_orders"]
    / len(shipped_orders)
    * 100
)


print("\nPROCESSING TIME DISTRIBUTION\n")

print(
    processing_distribution.to_string(
        index=False,
        formatters={
            "order_share_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

ORDER-TO-SHIPMENT PROCESSING TIME

Shipped Orders              : 24,798
Average Processing Days     : 3.52
Median Processing Days      : 4.00
Minimum Processing Days     : 1
Maximum Processing Days     : 6

PROCESSING TIME DISTRIBUTION

processing_time_band  total_orders order_share_pct
            Same Day             0           0.00%
               1 Day          4039          16.29%
              2 Days          4153          16.75%
              3 Days          4102          16.54%
            4-5 Days          8315          33.53%
             6+ Days          4189          16.89%


## Step 17 — Order-to-Shipment Processing Time by Priority

Analyze warehouse processing speed across order-priority levels.

The analysis compares:

- Shipped Orders
- Average Processing Days
- Median Processing Days
- Orders Shipped Within 2 Days
- Orders Taking 4 or More Days
- Fast Processing Rate
- Slow Processing Rate

This determines whether operational execution aligns with order priority.

In [17]:
# Order-to-shipment processing performance by priority

priority_processing = (
    shipped_orders
    .groupby("priority", as_index=False)
    .agg(
        shipped_orders=("order_id", "nunique"),
        avg_processing_days=(
            "order_to_shipment_days",
            "mean"
        ),
        median_processing_days=(
            "order_to_shipment_days",
            "median"
        )
    )
)


# Orders shipped within 2 days

fast_orders = (
    shipped_orders[
        shipped_orders["order_to_shipment_days"] <= 2
    ]
    .groupby("priority")["order_id"]
    .nunique()
)


priority_processing["within_2_days"] = (
    priority_processing["priority"]
    .map(fast_orders)
    .fillna(0)
    .astype(int)
)


# Orders taking 4+ days

slow_orders = (
    shipped_orders[
        shipped_orders["order_to_shipment_days"] >= 4
    ]
    .groupby("priority")["order_id"]
    .nunique()
)


priority_processing["four_plus_days"] = (
    priority_processing["priority"]
    .map(slow_orders)
    .fillna(0)
    .astype(int)
)


# Processing rates

priority_processing["within_2_days_pct"] = (
    priority_processing["within_2_days"]
    / priority_processing["shipped_orders"]
    * 100
)


priority_processing["four_plus_days_pct"] = (
    priority_processing["four_plus_days"]
    / priority_processing["shipped_orders"]
    * 100
)


# Sort fastest to slowest

priority_processing = (
    priority_processing
    .sort_values(
        "avg_processing_days"
    )
    .reset_index(drop=True)
)


print("ORDER-TO-SHIPMENT PROCESSING BY PRIORITY\n")

print(
    priority_processing.to_string(
        index=False,
        formatters={
            "avg_processing_days":
                lambda x: f"{x:.2f}",
            "median_processing_days":
                lambda x: f"{x:.2f}",
            "within_2_days_pct":
                lambda x: f"{x:.2f}%",
            "four_plus_days_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

ORDER-TO-SHIPMENT PROCESSING BY PRIORITY

priority  shipped_orders avg_processing_days median_processing_days  within_2_days  four_plus_days within_2_days_pct four_plus_days_pct
    High            6299                3.50                   4.00           2089            3155            33.16%             50.09%
     Low            2021                3.50                   3.00            678            1005            33.55%             49.73%
  Normal           14448                3.52                   4.00           4786            7290            33.13%             50.46%
  Urgent            2030                3.56                   4.00            639            1054            31.48%             51.92%


## Step 18 — Delivery & OTIF Performance by Fulfillment Status

Evaluate downstream delivery performance for different order-fulfillment outcomes.

Only shipments with valid chronology are included in delivery KPI calculations.

The analysis compares:

- Valid Delivered Orders
- On-Time Delivery Rate
- OTIF Rate
- Average Delivery Delay
- Fully Fulfilled Orders
- Partially Fulfilled / Backlogged Orders

This helps determine whether upstream fulfillment problems also translate into weaker customer delivery performance.

In [18]:
# Keep only shipments with valid delivery chronology

valid_delivery_orders = order_fulfillment[
    order_fulfillment["chronology_valid_flag"] == "Y"
].copy()


# Delivery performance by fulfillment status

delivery_by_fulfillment = (
    valid_delivery_orders
    .groupby(
        "fulfillment_status",
        as_index=False
    )
    .agg(
        valid_delivered_orders=("order_id", "nunique"),
        avg_order_fill_rate_pct=("order_fill_rate_pct", "mean"),
        avg_delivery_delay_days=(
            "delivery_delay_days_clean",
            "mean"
        ),
        on_time_rate_pct=(
            "on_time_flag_clean",
            lambda x: (x == "Y").mean() * 100
        ),
        otif_rate_pct=(
            "otif_flag_clean",
            lambda x: (x == "Y").mean() * 100
        )
    )
)


delivery_by_fulfillment = (
    delivery_by_fulfillment
    .sort_values(
        "otif_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "DELIVERY & OTIF PERFORMANCE BY FULFILLMENT STATUS\n"
)

print(
    delivery_by_fulfillment.to_string(
        index=False,
        formatters={
            "avg_order_fill_rate_pct":
                lambda x: f"{x:.2f}%",
            "avg_delivery_delay_days":
                lambda x: f"{x:.2f}",
            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",
            "otif_rate_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


print(
    f"\nValid Delivery Orders : "
    f"{len(valid_delivery_orders):,}"
)

DELIVERY & OTIF PERFORMANCE BY FULFILLMENT STATUS

              fulfillment_status  valid_delivered_orders avg_order_fill_rate_pct avg_delivery_delay_days on_time_rate_pct otif_rate_pct
                 Fully Fulfilled                   11323                 100.00%                    0.15           83.05%        83.05%
Partially Fulfilled / Backlogged                    9008                  72.38%                    1.92           56.35%         0.00%

Valid Delivery Orders : 20,331


## Step 19 — Quantify Fulfillment Impact on On-Time Delivery

Quantify the relationship between upstream order fulfillment and downstream on-time delivery performance.

The analysis measures:

- On-Time Rate for Fully Fulfilled Orders
- On-Time Rate for Backlogged Orders
- On-Time Performance Gap
- Late Delivery Rate
- Late Order Counts
- Relative Late-Delivery Risk

This provides a direct measure of how strongly incomplete fulfillment is associated with customer delivery failure.

In [19]:
# On-time and late-delivery performance by fulfillment status

fulfillment_delivery_risk = (
    valid_delivery_orders
    .groupby(
        "fulfillment_status",
        as_index=False
    )
    .agg(
        total_orders=("order_id", "nunique"),
        on_time_orders=(
            "on_time_flag_clean",
            lambda x: (x == "Y").sum()
        ),
        late_orders=(
            "on_time_flag_clean",
            lambda x: (x == "N").sum()
        )
    )
)


fulfillment_delivery_risk["on_time_rate_pct"] = (
    fulfillment_delivery_risk["on_time_orders"]
    / fulfillment_delivery_risk["total_orders"]
    * 100
)


fulfillment_delivery_risk["late_rate_pct"] = (
    fulfillment_delivery_risk["late_orders"]
    / fulfillment_delivery_risk["total_orders"]
    * 100
)


print("FULFILLMENT IMPACT ON DELIVERY PERFORMANCE\n")

print(
    fulfillment_delivery_risk.to_string(
        index=False,
        formatters={
            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",
            "late_rate_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


# Extract rates for comparison

fully_late_rate = fulfillment_delivery_risk.loc[
    fulfillment_delivery_risk[
        "fulfillment_status"
    ] == "Fully Fulfilled",
    "late_rate_pct"
].iloc[0]


backlogged_late_rate = fulfillment_delivery_risk.loc[
    fulfillment_delivery_risk[
        "fulfillment_status"
    ] == "Partially Fulfilled / Backlogged",
    "late_rate_pct"
].iloc[0]


fully_on_time_rate = fulfillment_delivery_risk.loc[
    fulfillment_delivery_risk[
        "fulfillment_status"
    ] == "Fully Fulfilled",
    "on_time_rate_pct"
].iloc[0]


backlogged_on_time_rate = fulfillment_delivery_risk.loc[
    fulfillment_delivery_risk[
        "fulfillment_status"
    ] == "Partially Fulfilled / Backlogged",
    "on_time_rate_pct"
].iloc[0]


on_time_gap = (
    fully_on_time_rate
    - backlogged_on_time_rate
)


relative_late_risk = (
    backlogged_late_rate
    / fully_late_rate
)


print(
    f"\nOn-Time Performance Gap       : "
    f"{on_time_gap:.2f} percentage points"
)

print(
    f"Relative Late-Delivery Risk   : "
    f"{relative_late_risk:.2f}x"
)

FULFILLMENT IMPACT ON DELIVERY PERFORMANCE

              fulfillment_status  total_orders  on_time_orders  late_orders on_time_rate_pct late_rate_pct
                 Fully Fulfilled         11323            9404         1919           83.05%        16.95%
Partially Fulfilled / Backlogged          9008            5076         3932           56.35%        43.65%

On-Time Performance Gap       : 26.70 percentage points
Relative Late-Delivery Risk   : 2.58x


## Step 20 — End-to-End Service Outcome Classification

Classify customer orders according to their complete fulfillment and delivery outcome.

Orders are grouped into operational service outcomes such as:

- Fully Fulfilled & On-Time
- Fully Fulfilled but Late
- Backlogged but On-Time
- Backlogged & Late
- Backlogged & Not Shipped
- Delivery Chronology Invalid

This provides a consolidated view of where customer-service failures occur across the complete order lifecycle.

In [20]:
# Create end-to-end service outcome

conditions = [

    # Fully fulfilled + valid + on time
    (
        (order_fulfillment["fulfillment_status"] == "Fully Fulfilled")
        & (order_fulfillment["chronology_valid_flag"] == "Y")
        & (order_fulfillment["on_time_flag_clean"] == "Y")
    ),

    # Fully fulfilled + valid + late
    (
        (order_fulfillment["fulfillment_status"] == "Fully Fulfilled")
        & (order_fulfillment["chronology_valid_flag"] == "Y")
        & (order_fulfillment["on_time_flag_clean"] == "N")
    ),

    # Backlogged + valid + on time
    (
        (order_fulfillment["fulfillment_status"]
         == "Partially Fulfilled / Backlogged")
        & (order_fulfillment["chronology_valid_flag"] == "Y")
        & (order_fulfillment["on_time_flag_clean"] == "Y")
    ),

    # Backlogged + valid + late
    (
        (order_fulfillment["fulfillment_status"]
         == "Partially Fulfilled / Backlogged")
        & (order_fulfillment["chronology_valid_flag"] == "Y")
        & (order_fulfillment["on_time_flag_clean"] == "N")
    ),

    # Backlogged and not shipped
    (
        (order_fulfillment["fulfillment_status"]
         == "Partially Fulfilled / Backlogged")
        & (order_fulfillment["shipment_status"] == "Not Shipped")
    ),

    # Shipped but chronology invalid
    (
        (order_fulfillment["shipment_status"] == "Shipped")
        & (order_fulfillment["chronology_valid_flag"] == "N")
    )
]


labels = [
    "Fully Fulfilled & On-Time",
    "Fully Fulfilled but Late",
    "Backlogged but On-Time",
    "Backlogged & Late",
    "Backlogged & Not Shipped",
    "Delivery Chronology Invalid"
]


order_fulfillment["end_to_end_service_outcome"] = np.select(
    conditions,
    labels,
    default="Review"
)


service_outcome_summary = (
    order_fulfillment
    .groupby(
        "end_to_end_service_outcome",
        as_index=False
    )
    .agg(
        total_orders=("order_id", "nunique"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum")
    )
)


service_outcome_summary["order_share_pct"] = (
    service_outcome_summary["total_orders"]
    / len(order_fulfillment)
    * 100
)


service_outcome_summary = (
    service_outcome_summary
    .sort_values(
        "total_orders",
        ascending=False
    )
    .reset_index(drop=True)
)


print("END-TO-END SERVICE OUTCOME SUMMARY\n")

print(
    service_outcome_summary.to_string(
        index=False,
        formatters={
            "order_share_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


print(
    "\nUnclassified / Review Orders :",
    (
        order_fulfillment[
            "end_to_end_service_outcome"
        ] == "Review"
    ).sum()
)

END-TO-END SERVICE OUTCOME SUMMARY

 end_to_end_service_outcome  total_orders  ordered_qty  fulfilled_qty  backlog_qty order_share_pct
  Fully Fulfilled & On-Time          9404       173655         173655            0          37.62%
     Backlogged but On-Time          5076       114699          86215        28484          20.30%
Delivery Chronology Invalid          4467        87950          78891         9059          17.87%
          Backlogged & Late          3932        88508          66543        21965          15.73%
   Fully Fulfilled but Late          1919        35272          35272            0           7.68%
   Backlogged & Not Shipped           202         1671              0         1671           0.81%

Unclassified / Review Orders : 0


## Step 21 — End-to-End Service Performance by Priority

Evaluate complete service performance across order-priority levels.

For valid delivery records, the analysis compares:

- Total Valid Orders
- Fully Fulfilled Orders
- Fully Fulfilled & On-Time Orders
- Backlogged & Late Orders
- End-to-End Success Rate
- Backlogged & Late Failure Rate

An end-to-end successful order is defined as an order that is:

**Fully Fulfilled + Shipped + Delivered On-Time**

This measures whether higher-priority orders actually receive stronger customer-service outcomes.

In [21]:
# Use only orders with valid delivery chronology

priority_service_base = order_fulfillment[
    order_fulfillment["chronology_valid_flag"] == "Y"
].copy()


# Create end-to-end success flag

priority_service_base["end_to_end_success_flag"] = np.where(
    (
        priority_service_base["fulfillment_status"]
        == "Fully Fulfilled"
    )
    &
    (
        priority_service_base["on_time_flag_clean"]
        == "Y"
    ),
    1,
    0
)


# Create backlogged & late failure flag

priority_service_base["backlogged_late_flag"] = np.where(
    (
        priority_service_base["fulfillment_status"]
        == "Partially Fulfilled / Backlogged"
    )
    &
    (
        priority_service_base["on_time_flag_clean"]
        == "N"
    ),
    1,
    0
)


# Priority-level service summary

priority_service = (
    priority_service_base
    .groupby("priority", as_index=False)
    .agg(
        valid_orders=("order_id", "nunique"),

        fully_fulfilled_orders=(
            "fulfillment_status",
            lambda x: (x == "Fully Fulfilled").sum()
        ),

        successful_orders=(
            "end_to_end_success_flag",
            "sum"
        ),

        backlogged_late_orders=(
            "backlogged_late_flag",
            "sum"
        )
    )
)


# End-to-end success rate

priority_service["end_to_end_success_rate_pct"] = (
    priority_service["successful_orders"]
    / priority_service["valid_orders"]
    * 100
)


# Backlogged & late failure rate

priority_service["backlogged_late_rate_pct"] = (
    priority_service["backlogged_late_orders"]
    / priority_service["valid_orders"]
    * 100
)


priority_service = (
    priority_service
    .sort_values(
        "end_to_end_success_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("END-TO-END SERVICE PERFORMANCE BY PRIORITY\n")

print(
    priority_service.to_string(
        index=False,
        formatters={
            "end_to_end_success_rate_pct":
                lambda x: f"{x:.2f}%",
            "backlogged_late_rate_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

END-TO-END SERVICE PERFORMANCE BY PRIORITY

priority  valid_orders  fully_fulfilled_orders  successful_orders  backlogged_late_orders end_to_end_success_rate_pct backlogged_late_rate_pct
     Low          1637                     935                803                     287                      49.05%                   17.53%
  Normal         11683                    6443               5594                    2105                      47.88%                   18.02%
    High          5264                    2964               2274                    1143                      43.20%                   21.71%
  Urgent          1747                     981                733                     397                      41.96%                   22.72%


## Step 22 — Identify High-Risk Customer Orders

Identify customer orders requiring the greatest operational attention.

A High-Risk Order is defined as a **High or Urgent priority order** that experiences at least one of the following:

- Remaining backlog
- Late delivery based on valid delivery chronology
- No shipment

This combines business priority with actual service failure to create an actionable exception list.

In [22]:
# Create high-risk order flag

priority_risk = order_fulfillment[
    "priority"
].isin(
    ["High", "Urgent"]
)


backlog_risk = (
    order_fulfillment["backlog_qty"] > 0
)


late_risk = (
    order_fulfillment["on_time_flag_clean"] == "N"
)


unshipped_risk = (
    order_fulfillment["shipment_status"] == "Not Shipped"
)


order_fulfillment["high_risk_order_flag"] = np.where(
    priority_risk
    & (
        backlog_risk
        | late_risk
        | unshipped_risk
    ),
    "Y",
    "N"
)


high_risk_orders = order_fulfillment[
    order_fulfillment["high_risk_order_flag"] == "Y"
].copy()


print("HIGH-RISK ORDER SUMMARY\n")

print(
    f"High-Risk Orders           : "
    f"{len(high_risk_orders):,}"
)

print(
    f"Share of All Orders        : "
    f"{len(high_risk_orders) / len(order_fulfillment) * 100:.2f}%"
)

print(
    f"High-Risk Ordered Qty      : "
    f"{high_risk_orders['ordered_qty'].sum():,.0f}"
)

print(
    f"High-Risk Backlog Qty      : "
    f"{high_risk_orders['backlog_qty'].sum():,.0f}"
)

print(
    f"High-Risk Backlog Value    : "
    f"{high_risk_orders['backlog_value'].sum():,.2f}"
)


print("\nHIGH-RISK ORDERS BY PRIORITY\n")

high_risk_priority_summary = (
    high_risk_orders
    .groupby("priority", as_index=False)
    .agg(
        high_risk_orders=("order_id", "nunique"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum")
    )
)


print(
    high_risk_priority_summary.to_string(
        index=False,
        formatters={
            "backlog_value":
                lambda x: f"{x:,.2f}"
        }
    )
)

HIGH-RISK ORDER SUMMARY

High-Risk Orders           : 4,549
Share of All Orders        : 18.20%
High-Risk Ordered Qty      : 97,658
High-Risk Backlog Qty      : 20,435
High-Risk Backlog Value    : 87,346,461.35

HIGH-RISK ORDERS BY PRIORITY

priority  high_risk_orders  ordered_qty  fulfilled_qty  backlog_qty backlog_value
    High              3425        73808          58595        15213 65,058,906.65
  Urgent              1124        23850          18628         5222 22,287,554.70


## Step 23 — High-Risk Orders by Warehouse

Analyze the concentration of High-Risk Orders across fulfillment warehouses.

The analysis compares:

- High-Risk Order Count
- High-Risk Ordered Quantity
- Fulfilled Quantity
- Backlog Quantity
- Backlog Value
- Share of Total High-Risk Orders

This identifies operational locations where priority-order service failures are most concentrated.

In [23]:
# High-risk order exposure by warehouse

high_risk_warehouse = (
    high_risk_orders
    .groupby("warehouse_id", as_index=False)
    .agg(
        high_risk_orders=("order_id", "nunique"),
        ordered_qty=("ordered_qty", "sum"),
        fulfilled_qty=("fulfilled_qty", "sum"),
        backlog_qty=("backlog_qty", "sum"),
        backlog_value=("backlog_value", "sum")
    )
)


# Add warehouse names

high_risk_warehouse = high_risk_warehouse.merge(
    warehouse_lookup,
    on="warehouse_id",
    how="left",
    validate="one_to_one"
)


# Share of all high-risk orders

high_risk_warehouse["high_risk_order_share_pct"] = (
    high_risk_warehouse["high_risk_orders"]
    / len(high_risk_orders)
    * 100
)


# Sort by backlog value

high_risk_warehouse = (
    high_risk_warehouse
    .sort_values(
        "backlog_value",
        ascending=False
    )
    .reset_index(drop=True)
)


# Arrange columns

high_risk_warehouse = high_risk_warehouse[
    [
        "warehouse_id",
        "warehouse_name",
        "high_risk_orders",
        "ordered_qty",
        "fulfilled_qty",
        "backlog_qty",
        "backlog_value",
        "high_risk_order_share_pct"
    ]
]


print("HIGH-RISK ORDER EXPOSURE BY WAREHOUSE\n")

print(
    high_risk_warehouse.to_string(
        index=False,
        formatters={
            "backlog_value":
                lambda x: f"{x:,.2f}",
            "high_risk_order_share_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

HIGH-RISK ORDER EXPOSURE BY WAREHOUSE

warehouse_id    warehouse_name  high_risk_orders  ordered_qty  fulfilled_qty  backlog_qty backlog_value high_risk_order_share_pct
        WH01 Bengaluru Central               982        21026          16660         4366 19,038,916.63                    21.59%
        WH03         Delhi NCR               873        18639          14781         3858 16,519,241.27                    19.19%
        WH02       Mumbai West               826        18355          14540         3815 15,974,954.41                    18.16%
        WH06           Pune DC               633        13636          10739         2897 12,596,801.65                    13.92%
        WH04   Hyderabad South               648        12779          10053         2726 11,929,675.35                    14.24%
        WH05       Chennai Hub               587        13223          10450         2773 11,286,872.04                    12.90%


## Step 24 — Order Fulfillment Analysis Reconciliation

Perform a comprehensive reconciliation of the Order Fulfillment Analysis.

The checks confirm consistency across:

- Order counts
- Ordered, fulfilled, and backlog quantities
- Shipment coverage
- Fulfilled vs shipped quantities
- Valid delivery records
- End-to-end service outcomes
- High-risk order identification

All reconciliation checks must pass before final business conclusions are documented.

In [24]:
# Order Fulfillment Analysis Reconciliation

reconciliation_checks = {

    "Total Order Count":
        len(order_fulfillment) == 25_000,

    "Unique Order Count":
        order_fulfillment["order_id"].nunique() == 25_000,

    "No Duplicate Order IDs":
        order_fulfillment["order_id"].duplicated().sum() == 0,

    "Total Ordered Quantity":
        np.isclose(
            order_fulfillment["ordered_qty"].sum(),
            501_755
        ),

    "Total Fulfilled Quantity":
        np.isclose(
            order_fulfillment["fulfilled_qty"].sum(),
            440_576
        ),

    "Total Backlog Quantity":
        np.isclose(
            order_fulfillment["backlog_qty"].sum(),
            61_179
        ),

    "Quantity Reconciliation":
        np.isclose(
            (
                order_fulfillment["fulfilled_qty"].sum()
                + order_fulfillment["backlog_qty"].sum()
            ),
            order_fulfillment["ordered_qty"].sum()
        ),

    "Fully Fulfilled Order Count":
        (
            order_fulfillment["backlog_qty"] == 0
        ).sum() == 14_150,

    "Backlogged Order Count":
        (
            order_fulfillment["backlog_qty"] > 0
        ).sum() == 10_850,

    "Shipped Order Count":
        (
            order_fulfillment["shipment_status"] == "Shipped"
        ).sum() == 24_798,

    "Unshipped Order Count":
        (
            order_fulfillment["shipment_status"] == "Not Shipped"
        ).sum() == 202,

    "Fulfilled vs Shipped Quantity":
        np.isclose(
            order_fulfillment["fulfilled_qty"].sum(),
            order_fulfillment["shipped_qty_filled"].sum()
        ),

    "Valid Delivery Order Count":
        (
            order_fulfillment["chronology_valid_flag"] == "Y"
        ).sum() == 20_331,

    "Invalid Delivery Chronology Count":
        (
            order_fulfillment["chronology_valid_flag"] == "N"
        ).sum() == 4_467,

    "End-to-End Outcome Complete":
        order_fulfillment[
            "end_to_end_service_outcome"
        ].notna().all(),

    "No Review Outcomes":
        (
            order_fulfillment[
                "end_to_end_service_outcome"
            ] == "Review"
        ).sum() == 0,

    "Service Outcome Count Reconciles":
        service_outcome_summary[
            "total_orders"
        ].sum() == 25_000,

    "High-Risk Order Count":
        len(high_risk_orders) == 4_549,

    "High-Risk Backlog Quantity":
        np.isclose(
            high_risk_orders["backlog_qty"].sum(),
            20_435
        ),

    "High-Risk Backlog Value":
        np.isclose(
            high_risk_orders["backlog_value"].sum(),
            87_346_461.35,
            atol=0.01
        )
}


print("ORDER FULFILLMENT ANALYSIS RECONCILIATION\n")


for check, result in reconciliation_checks.items():

    print(
        f"{check:<45} : "
        f"{'PASS' if result else 'FAIL'}"
    )


order_fulfillment_reconciliation_passed = (
    all(reconciliation_checks.values())
)


print(
    "\nORDER FULFILLMENT RECONCILIATION STATUS :",
    "PASSED"
    if order_fulfillment_reconciliation_passed
    else "REVIEW REQUIRED"
)

ORDER FULFILLMENT ANALYSIS RECONCILIATION

Total Order Count                             : PASS
Unique Order Count                            : PASS
No Duplicate Order IDs                        : PASS
Total Ordered Quantity                        : PASS
Total Fulfilled Quantity                      : PASS
Total Backlog Quantity                        : PASS
Quantity Reconciliation                       : PASS
Fully Fulfilled Order Count                   : PASS
Backlogged Order Count                        : PASS
Shipped Order Count                           : PASS
Unshipped Order Count                         : PASS
Fulfilled vs Shipped Quantity                 : PASS
Valid Delivery Order Count                    : PASS
Invalid Delivery Chronology Count             : PASS
End-to-End Outcome Complete                   : PASS
No Review Outcomes                            : PASS
Service Outcome Count Reconciles              : PASS
High-Risk Order Count                         : PASS
Hig

## Step 25 — Key Business Insights

### 1. Overall Fulfillment Performance

Across 25,000 customer orders:

- Total Ordered Quantity: **501,755 units**
- Total Fulfilled Quantity: **440,576 units**
- Total Backlog Quantity: **61,179 units**
- Overall Quantity Fill Rate: **87.81%**
- Average Order Fill Rate: **87.44%**

A total of **14,150 orders (56.60%)** were fully fulfilled, while **10,850 orders (43.40%)** contained some backlog.

This indicates that although most ordered units are fulfilled, partial fulfillment affects a substantial share of customer orders.

---

### 2. Fulfillment and Shipment Quantities Reconcile Completely

The dataset contains:

- **440,576 fulfilled units**
- **440,576 shipped units**
- **0 net quantity difference**

All 25,000 orders reconcile exactly between fulfilled and shipped quantity.

This confirms that the fulfillment quantity represents inventory that successfully progressed into shipment execution.

---

### 3. Shipment Coverage Is Very High

Of the 25,000 customer orders:

- **24,798 orders were shipped**
- **202 orders were not shipped**
- Overall shipment coverage is **99.19%**

All **14,150 fully fulfilled orders were shipped**.

The 202 unshipped orders were all backlogged orders with:

- **0 fulfilled units**
- **1,671 backlog units**

This indicates that these orders represent genuine fulfillment failures rather than missing shipment records.

---

### 4. Order Priority Is Not Producing Better Fulfillment

Quantity fill rates by priority were:

- High: **88.23%**
- Normal: **87.82%**
- Low: **87.18%**
- Urgent: **86.97%**

Urgent orders have the lowest quantity fill rate despite carrying the highest operational priority.

This suggests that priority designation alone is not sufficient to overcome inventory or fulfillment constraints.

---

### 5. Urgent Orders Are Also Processed the Slowest

Average order-to-shipment processing time is **3.52 days**.

Urgent orders average **3.56 days**, compared with approximately **3.50 days** for High and Low priority orders.

Only **31.48% of Urgent orders** are shipped within two days, while **51.92% take four or more days**.

This indicates that the priority system is not strongly accelerating warehouse execution.

---

### 6. Sales Channel Is Not a Major Fulfillment Driver

Quantity fill rates across sales channels remain tightly grouped:

- Distributor: **88.48%**
- Direct: **87.73%**
- Online: **87.56%**
- Marketplace: **87.54%**

The limited variation suggests that fulfillment problems are not primarily driven by the sales channel.

---

### 7. Customer Segment Differences Are Also Limited

Distributor customers achieve the strongest quantity fill rate at **88.24%**, while Enterprise customers are lowest at **87.58%**.

Retail has the highest backlog-order rate at **43.68%**.

However, the relatively narrow performance range indicates that customer segment alone does not explain the broader fulfillment problem.

---

### 8. Regional Fulfillment Risk Is More Visible

Regional quantity fill rates show greater variation:

- West: **88.44%**
- East: **88.04%**
- South: **88.02%**
- Central: **87.19%**
- North: **86.90%**

North has both the lowest quantity fill rate and the highest backlog-order rate at **44.59%**.

This makes North an important region for further inventory-allocation and service-level investigation.

---

### 9. Warehouse Performance Shows Localized Fulfillment Pressure

Mumbai West has the strongest quantity fill rate at **88.27%**, followed by Delhi NCR at **88.19%**.

Pune DC and Chennai Hub have the lowest quantity fill rates at approximately **87.12%**.

Pune DC also has the highest backlog-order rate at **44.15%**.

These differences suggest localized warehouse-level inventory or execution constraints.

---

### 10. Automotive Is the Weakest Product Category

Product-category analysis shows:

- Home & Kitchen: **88.84% quantity fill rate**
- Office Supplies: **88.02%**
- Electronics: **87.91%**
- Health & Personal Care: **87.49%**
- Automotive: **86.63%**

Automotive also has the highest backlog-line rate at **23.01%**.

This makes Automotive the clearest product-category fulfillment risk.

---

### 11. Backlog Strongly Affects Delivery Performance

Among orders with valid delivery chronology:

Fully fulfilled orders achieve:

- **83.05% On-Time Delivery**
- **83.05% OTIF**
- **0.15 average delay days**

Partially fulfilled / backlogged orders achieve:

- **56.35% On-Time Delivery**
- **0% OTIF**
- **1.92 average delay days**

The 0% OTIF rate for backlogged orders is expected because an order cannot satisfy the "In Full" requirement while quantity remains backlogged.

---

### 12. Backlogged Orders Have 2.58× Higher Late-Delivery Risk

Fully fulfilled orders have a late-delivery rate of only **16.95%**.

Backlogged orders have a late-delivery rate of **43.65%**.

This creates:

- **26.70 percentage-point On-Time performance gap**
- **2.58× relative late-delivery risk**

This demonstrates a strong operational association between incomplete fulfillment and downstream delivery failure.

---

### 13. End-to-End Service Outcomes

Across all 25,000 orders:

- **9,404 (37.62%)** — Fully Fulfilled & On-Time
- **5,076 (20.30%)** — Backlogged but On-Time
- **3,932 (15.73%)** — Backlogged & Late
- **1,919 (7.68%)** — Fully Fulfilled but Late
- **202 (0.81%)** — Backlogged & Not Shipped
- **4,467 (17.87%)** — Delivery Chronology Invalid

The chronology-invalid records are kept separate so they do not distort valid delivery-performance conclusions.

---

### 14. Priority Is Not Translating Into End-to-End Service Success

For orders with valid delivery chronology, end-to-end success rates are:

- Low: **49.05%**
- Normal: **47.88%**
- High: **43.20%**
- Urgent: **41.96%**

Urgent orders also have the highest Backlogged & Late rate at **22.72%**.

This is a major operational finding: higher business priority is associated with weaker rather than stronger end-to-end service outcomes in this dataset.

---

### 15. High-Risk Priority Orders Require Immediate Attention

A total of **4,549 High/Urgent orders** were classified as high risk because they experienced backlog, late delivery, or no shipment.

These represent:

- **18.20% of all orders**
- **97,658 ordered units**
- **20,435 backlog units**
- Approximately **87.35 million in backlog value**

High-priority orders account for most of this exposure:

- High: **3,425 high-risk orders**
- Urgent: **1,124 high-risk orders**

This provides a focused exception population for operational intervention.

---

### 16. Bengaluru Central Has the Largest High-Risk Warehouse Exposure

Bengaluru Central contains:

- **982 high-risk orders**
- **4,366 backlog units**
- Approximately **19.04 million in backlog value**
- **21.59% of all high-risk orders**

Delhi NCR and Mumbai West follow with approximately **16.52 million** and **15.97 million** in high-risk backlog value respectively.

This indicates that priority-order service risk is concentrated unevenly across warehouses.

---

## Management Takeaway

The primary customer-service problem is not shipment coverage: **99.19% of orders progress to shipment**, and fulfilled quantities reconcile exactly with shipped quantities.

The larger issue occurs earlier in the process through **incomplete order fulfillment**.

Backlogged orders are substantially more likely to arrive late, and incomplete fulfillment eliminates the possibility of achieving OTIF.

The analysis also reveals that the current priority system is not producing the intended operational outcome. Urgent orders have the weakest quantity fulfillment, slowest average processing, lowest valid end-to-end success rate, and highest Backlogged & Late rate.

Management attention should therefore focus on:

1. Improving inventory availability before orders enter backlog
2. Reviewing how High and Urgent orders are prioritized operationally
3. Investigating North-region fulfillment constraints
4. Addressing Automotive category shortages
5. Reviewing Pune DC's backlog performance
6. Prioritizing the **4,549 identified high-risk High/Urgent orders**
7. Investigating Bengaluru Central's **19.04 million high-risk backlog exposure**
8. Continuing to exclude chronology-invalid delivery records from operational delivery KPIs until the underlying data-quality issue is resolved

The strongest opportunity is to improve **upstream inventory fulfillment and priority-order execution**, because these factors directly influence downstream On-Time and OTIF service performance.

## Step 26 — Final Order Fulfillment Validation

Perform the final validation of the Order Fulfillment Analysis notebook.

The validation confirms that:

- All 25,000 customer orders are preserved
- Order IDs remain unique
- Ordered quantity reconciles with fulfilled and backlog quantities
- Fulfilled quantity reconciles exactly with shipped quantity
- Shipment coverage is complete and consistent
- Valid and invalid delivery records reconcile
- Customer, region, warehouse, and product mappings are complete
- End-to-end service outcomes cover every order
- High-risk order identification reconciles
- No negative fulfillment quantities exist
- Step 24 reconciliation passed

This ensures that the notebook is analytically consistent and ready for downstream reporting and business recommendations.

In [25]:
# Final Order Fulfillment Analysis Validation

valid_service_outcomes = {
    "Fully Fulfilled & On-Time",
    "Fully Fulfilled but Late",
    "Backlogged but On-Time",
    "Backlogged & Late",
    "Backlogged & Not Shipped",
    "Delivery Chronology Invalid"
}


final_validation_checks = {

    "Order Record Count":
        len(order_fulfillment) == 25_000,

    "Unique Order Count":
        order_fulfillment["order_id"].nunique() == 25_000,

    "No Duplicate Order IDs":
        order_fulfillment["order_id"].duplicated().sum() == 0,

    "Ordered Quantity Reconciles":
        np.isclose(
            order_fulfillment["ordered_qty"].sum(),
            501_755
        ),

    "Fulfilled Quantity Reconciles":
        np.isclose(
            order_fulfillment["fulfilled_qty"].sum(),
            440_576
        ),

    "Backlog Quantity Reconciles":
        np.isclose(
            order_fulfillment["backlog_qty"].sum(),
            61_179
        ),

    "Fulfilled Plus Backlog Equals Ordered":
        np.isclose(
            (
                order_fulfillment["fulfilled_qty"].sum()
                + order_fulfillment["backlog_qty"].sum()
            ),
            order_fulfillment["ordered_qty"].sum()
        ),

    "No Negative Ordered Quantity":
        (
            order_fulfillment["ordered_qty"] >= 0
        ).all(),

    "No Negative Fulfilled Quantity":
        (
            order_fulfillment["fulfilled_qty"] >= 0
        ).all(),

    "No Negative Backlog Quantity":
        (
            order_fulfillment["backlog_qty"] >= 0
        ).all(),

    "Fully Fulfilled Count":
        (
            order_fulfillment["fulfillment_status"]
            == "Fully Fulfilled"
        ).sum() == 14_150,

    "Backlogged Order Count":
        (
            order_fulfillment["fulfillment_status"]
            == "Partially Fulfilled / Backlogged"
        ).sum() == 10_850,

    "Shipped Order Count":
        (
            order_fulfillment["shipment_status"]
            == "Shipped"
        ).sum() == 24_798,

    "Unshipped Order Count":
        (
            order_fulfillment["shipment_status"]
            == "Not Shipped"
        ).sum() == 202,

    "Fulfilled Equals Shipped Quantity":
        np.isclose(
            order_fulfillment["fulfilled_qty"].sum(),
            order_fulfillment["shipped_qty_filled"].sum()
        ),

    "Valid Delivery Count":
        (
            order_fulfillment["chronology_valid_flag"]
            == "Y"
        ).sum() == 20_331,

    "Invalid Delivery Count":
        (
            order_fulfillment["chronology_valid_flag"]
            == "N"
        ).sum() == 4_467,

    "Delivery Records Reconcile":
        (
            (
                order_fulfillment["chronology_valid_flag"]
                == "Y"
            ).sum()
            +
            (
                order_fulfillment["chronology_valid_flag"]
                == "N"
            ).sum()
            +
            (
                order_fulfillment["shipment_status"]
                == "Not Shipped"
            ).sum()
        ) == 25_000,

    "Customer Segment Complete":
        order_fulfillment[
            "customer_segment"
        ].notna().all(),

    "Customer Region Complete":
        order_fulfillment[
            "customer_region"
        ].notna().all(),

    "Product Category Mapping Complete":
        item_fulfillment[
            "category"
        ].notna().all(),

    "Service Outcome Complete":
        order_fulfillment[
            "end_to_end_service_outcome"
        ].notna().all(),

    "Service Outcome Domain Valid":
        set(
            order_fulfillment[
                "end_to_end_service_outcome"
            ].unique()
        ).issubset(
            valid_service_outcomes
        ),

    "No Review Service Outcomes":
        (
            order_fulfillment[
                "end_to_end_service_outcome"
            ] == "Review"
        ).sum() == 0,

    "Service Outcome Count Reconciles":
        service_outcome_summary[
            "total_orders"
        ].sum() == 25_000,

    "High-Risk Order Count":
        len(high_risk_orders) == 4_549,

    "High-Risk Backlog Quantity":
        np.isclose(
            high_risk_orders[
                "backlog_qty"
            ].sum(),
            20_435
        ),

    "High-Risk Backlog Value":
        np.isclose(
            high_risk_orders[
                "backlog_value"
            ].sum(),
            87_346_461.35,
            atol=0.01
        ),

    "Step 24 Reconciliation Passed":
        order_fulfillment_reconciliation_passed
}


print("FINAL ORDER FULFILLMENT VALIDATION\n")


for check, result in final_validation_checks.items():

    print(
        f"{check:<45} : "
        f"{'PASS' if result else 'FAIL'}"
    )


final_order_fulfillment_status = (
    all(final_validation_checks.values())
)


print(
    "\nFINAL ORDER FULFILLMENT ANALYSIS STATUS :",
    "PASSED"
    if final_order_fulfillment_status
    else "REVIEW REQUIRED"
)

FINAL ORDER FULFILLMENT VALIDATION

Order Record Count                            : PASS
Unique Order Count                            : PASS
No Duplicate Order IDs                        : PASS
Ordered Quantity Reconciles                   : PASS
Fulfilled Quantity Reconciles                 : PASS
Backlog Quantity Reconciles                   : PASS
Fulfilled Plus Backlog Equals Ordered         : PASS
No Negative Ordered Quantity                  : PASS
No Negative Fulfilled Quantity                : PASS
No Negative Backlog Quantity                  : PASS
Fully Fulfilled Count                         : PASS
Backlogged Order Count                        : PASS
Shipped Order Count                           : PASS
Unshipped Order Count                         : PASS
Fulfilled Equals Shipped Quantity             : PASS
Valid Delivery Count                          : PASS
Invalid Delivery Count                        : PASS
Delivery Records Reconcile                    : PASS
Customer S